# Open AI  API 學習之路

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 如何將我的API 叫進來

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

client = OpenAI()

response = client.responses.create(
    model="gpt-5",
    input="請用繁體中文介紹 NVIDIA",
    instructions="請用創辦人的口吻介紹 NVIDIA",
)

print(response.output_text)




True

API key loaded: True
以下內容為以 NVIDIA 創辦人口吻撰寫的創作性介紹，非真實引言。

1993 年，我們創立了 NVIDIA，帶著一個簡單而堅定的信念：加速運算會改變世界。當年我們發明了 GPU，讓影像與圖形栩栩如生；此後我們做了一個關鍵決定——讓 GPU 不只畫圖，而是成為通用的平行運算引擎。2006 年，我們推出 CUDA，把 GPU 的平行能力開放給全球開發者，這一步，為現今的高效能運算與人工智慧奠下基礎。

今天，NVIDIA 是一家以平台為核心的加速運算公司。我們不只打造晶片，更是提供從 GPU、CPU 與加速系統，到網路、軟體與整套開發工具的完整堆疊：
- 加速運算與資料中心：從 Tensor Core GPU（如 A100、H100、H200）到基於 Blackwell 架構的新一代平台，結合 NVLink、NVSwitch 與我們在 InfiniBand/Ethernet 網路（含 Mellanox 技術）的佈局，為生成式 AI、科學運算與雲端提供可擴展的效能與能效。
- 軟體與生態系：CUDA、cuDNN、TensorRT、NCCL、CUDA-X 等庫與 SDK，讓數百萬開發者能把演算法直接映射到平行硬體上；DGX、AI Enterprise 與主要雲端服務商的合作，讓企業能更快把模型推向生產環境。
- 產業平台：GeForce 與 RTX 推動即時光線追蹤與 AI 圖形；DRIVE 服務自動駕駛與車載運算；Omniverse 與數位孿生幫助工業與內容創作；Isaac 讓機器人從模擬到實體協同學習；在醫療、金融、零售、能源等行業，我們與夥伴一起把資料轉化為可運行的智慧。

我們相信，未來的每家公司都是 AI 公司；每個資料中心都將成為「AI 工廠」，把資料轉為模型、把模型轉為應用。加速運算不僅追求速度，更追求能效與規模，讓科學家、工程師與創作者用更少能源完成更大的計算，從藥物設計到氣候模擬、從推薦系統到多模態生成式 AI。

NVIDIA 的文化很簡單：長期主義、以工程為先、與生態系共創。我們每天問自己兩個問題：還有哪些計算瓶頸沒有被移除？我們能否把它做成平台，讓全世界的開發者站在更高的起點上？如果你關心效能、效率與影響力，歡迎加入我們的旅程。


# 使用 stream 讓訊息可以一步一步輸出

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

client = OpenAI()

response = client.responses.create(
    model="gpt-5",
    input="請用繁體中文介紹 NVIDIA",
    instructions="請用創辦人的口吻介紹 NVIDIA",
    stream=True, ##流式輸出
)

for event in response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")

# 不同模型的用法 限制token數 以及設定角色

In [ ]:
response = client.responses.create(
    model="gpt-5",
    input="唧唧復唧唧有幾隻雞",
    # instructions 不設定會干擾模型的回答，因為模型會嘗試去滿足 instructions 的要求，但這裡我們只是想要模型回答問題，所以不設定 instructions。
    stream=True, ##流式輸出
    reasoning={"effort": "low"} ##意思是將回答的努力程度設定為低，可以省token，讓模型更快地回答問題 但錯誤率有點高(用high會更慢但更準確)
)

for event in response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")

In [ ]:
import base64

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

response = client.responses.create(
    model="gpt-5",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "請幫我分別描述這張兩張圖片",                   
                },

                {
                    "type": "input_image",
                    "image_url":f"data:image/png;base64,{encode_image('C:\\my_Purdue\\my_openAI_API\\procedure_group.png')}"
                },
            
            {
                "type": "input_image",
                "image_url": "https://www.i-meihua.com/FileUploads/ImediaArticlePhoto/61e963e3-d6f0-42c5-8742-a6c337d50543.jpg"
            }
            ]
        }

    ],
    # instructions 不設定會干擾模型的回答，因為模型會嘗試去滿足 instructions 的要求，但這裡我們只是想要模型回答問題，所以不設定 instructions。
    stream=True, ##流式輸出
    reasoning={"effort": "low"} ##意思是將回答的努力程度設定為低，可以省token，讓模型更快地回答問題 但錯誤率有點高(用high會更慢但更準確)
)

for event in response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")

# 要讓open AI 記得我們前面的對話 

In [ ]:
def chat():

    current_response_id = None

    while True:

        print("-----------------------------")
        print("當前id是", current_response_id)

        user_input = input("請輸入問題: ")

        if user_input == "退出":
            print("再見")
            break

        response = client.responses.create(
            model="gpt-5",
            input=user_input,
            previous_response_id=current_response_id
        )

        current_response_id = response.id

        print("AI回答:", response.output_text)
        print("新獲取的id是", current_response_id)

chat()



# 提取內容中的文字要素

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions="提取句子中的要素",
    input=[
        {
            "role": "user",
            "content": "1922年，尼克·卡拉威在紐約遇見了蓋茨比。"
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "timePlacePerson",
            "schema": {
                "type": "object",
                "properties": {
                    "time": {
                        "type": "number"
                    },
                    "place": {
                        "type": "string"
                    },
                    "person": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": ["time", "place", "person"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)

print(response.output_text)
                

# 提取圖片中的內容

In [ ]:
import base64
from typing import List
from openai import OpenAI
from pydantic import BaseModel,ConfigDict
## ConfigDict 就像是additionalProperties 規定AI 不能夠自己亂加東西

client = OpenAI()


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


base64_image = encode_image(r"C:\my_Purdue\my_openAI_API\procedure_group.png")

class OrderItem(BaseModel):
    item_name: str
    item_count: float
    item_price: float


class Order(BaseModel):
    order_items: List[OrderItem]


model_config = ConfigDict(
    extra="forbid"
)

response = client.responses.parse(
    model="gpt-4.1",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "請擷取圖片中的商品資訊，包含商品名稱、數量、價格。"
                },
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ],
    text_format=Order
)


order = response.output_parsed

print(order)

for item in order.order_items:
    print("商品名稱：", item.item_name)
    print("數量：", item.item_count)
    print("價格：", item.item_price)
    print("----------------")